##### Notebook 3 — School-to-Station Distance Join. This is where every one of your 2,425 schools gets matched to its nearest pollution monitoring station, using real-world distance.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.neighbors import BallTree

SCHOOL_PATH = Path("../data/raw/school_data.csv")
STATS_PATH = Path("../data/processed/station_pollution_stats.csv")
MASTER_PATH = Path("../data/processed/stations_master.csv")
OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
schools = pd.read_csv(SCHOOL_PATH)
stations = pd.read_csv(MASTER_PATH)
station_stats = pd.read_csv(STATS_PATH)

print("Schools:", schools.shape)
print("Stations:", stations.shape)
schools.head()

Schools: (2425, 7)
Stations: (45, 5)


,District,School Name,Address,SchoolLevel,Latitude,Longitude,Type
0,North West A,The Vivekanand School,"Shiv Mandir Colony, Narela, Delhi-40",Sr.Secondary,28.870837,77.080305,Unaided
1,North West A,NATIONAL PUBLIC SCHOOL,"SAFIABAD ROAD NARELA, DELHI - 110040",Secondary,28.863631,77.085591,Unaided
2,North West A,Singhu-G(Co-Ed)SSS,"Singhu,Delhi",Sr.Secondary,28.861792,77.136931,Govt
3,North West A,"Narela,Sect. A-6, Pkt-2 -SBV","GBSSS, Sect. A-6, Pkt-2, Narela, Delhi-110040",Sarvodaya Sr.Secondary,28.856882,77.100818,Govt
4,North West A,DAV CENTENARY PUBLIC SCHOOL,NARELA DELHI - 110040,Sr.Secondary,28.856308,77.084533,Unaided


In [4]:
before = len(schools)
schools = schools.dropna(subset=["Latitude", "Longitude"]).copy()

schools = schools[
    (schools["Latitude"].between(28.3, 28.95)) &
    (schools["Longitude"].between(76.8, 77.4))
].copy()

after = len(schools)
print(f"Dropped {before - after} schools with missing/invalid coordinates")
print(f"Remaining schools: {after}")

Dropped 2 schools with missing/invalid coordinates
Remaining schools: 2423


#####  The Haversine Distance Logic


In [11]:
def haversine_balltree_join(schools_df, stations_df):
    school_coords_rad = np.radians(schools_df[["Latitude", "Longitude"]].values)
    station_coords_rad = np.radians(stations_df[["latitude", "longitude"]].values)

    tree = BallTree(station_coords_rad, metric="haversine")
    distances, indices = tree.query(school_coords_rad, k=1)

    EARTH_RADIUS_KM = 6371.0
    distances_km = distances.flatten() * EARTH_RADIUS_KM
    nearest_station_idx = indices.flatten()

    return nearest_station_idx, distances_km

nearest_idx, distances_km = haversine_balltree_join(schools, stations)

schools["nearest_station_id"] = stations.iloc[nearest_idx]["station_id"].values
schools["nearest_station_name"] = stations.iloc[nearest_idx]["station_name"].values
schools["distance_km"] = distances_km.round(3)

schools[["School Name", "nearest_station_name", "distance_km"]].head(10)

,School Name,nearest_station_name,distance_km
0,The Vivekanand School,Narela,5.740
1,NATIONAL PUBLIC SCHOOL,Narela,4.809
2,Singhu-G(Co-Ed)SSS,Alipur,5.399
3,"Narela,Sect. A-6, Pkt-2 -SBV",Narela,3.787
4,DAV CENTENARY PUBLIC SCHOOL,Narela,4.092
5,"Narela, Sect. A-6, Pocket-2- GGSSS",Narela,3.113
6,Bankner-SBV,Narela,4.166
7,Bankner-GGSSS,Narela,4.160
8,NEW HAPPY PUBLIC SCHOOL,Narela,3.103
9,"Narela, Pocket 5 & 6-GGSSS",Narela,2.930


#####  Attach Pollution Exposure Stats to Each School

In [12]:
station_stats = station_stats.drop_duplicates(subset=["station_id"]).copy()

schools_with_exposure = schools.merge(
    station_stats,
    left_on="nearest_station_id",
    right_on="station_id",
    how="left",
    validate="m:1"
)

print(schools_with_exposure.shape)
schools_with_exposure[["School Name", "nearest_station_name", "distance_km", "PM2.5 (µg/m³)_mean"]].head(10)

(2423, 60)


,School Name,nearest_station_name,distance_km,PM2.5 (µg/m³)_mean
0,The Vivekanand School,Narela,5.740,109.104000
1,NATIONAL PUBLIC SCHOOL,Narela,4.809,109.104000
2,Singhu-G(Co-Ed)SSS,Alipur,5.399,99.947233
3,"Narela,Sect. A-6, Pkt-2 -SBV",Narela,3.787,109.104000
4,DAV CENTENARY PUBLIC SCHOOL,Narela,4.092,109.104000
5,"Narela, Sect. A-6, Pocket-2- GGSSS",Narela,3.113,109.104000
6,Bankner-SBV,Narela,4.166,109.104000
7,Bankner-GGSSS,Narela,4.160,109.104000
8,NEW HAPPY PUBLIC SCHOOL,Narela,3.103,109.104000
9,"Narela, Pocket 5 & 6-GGSSS",Narela,2.930,109.104000


In [13]:
def confidence_flag(km):
    if km <= 5:
        return "High"
    elif km <= 10:
        return "Medium"
    else:
        return "Low"

schools_with_exposure["exposure_confidence"] = schools_with_exposure["distance_km"].apply(confidence_flag)

print(schools_with_exposure["exposure_confidence"].value_counts())

exposure_confidence
High      2260
Medium     154
Low          9
Name: count, dtype: int64


In [15]:
print("Schools with no nearest station match:", schools_with_exposure["nearest_station_id"].isna().sum())
print("Max distance to nearest station (km):", schools_with_exposure["distance_km"].max())
print("Mean distance to nearest station (km):", schools_with_exposure["distance_km"].mean().round(2))
print("\nSchools farthest from any station:")
print(schools_with_exposure.nlargest(5, "distance_km")[["School Name", "District", "nearest_station_name", "distance_km"]])

Schools with no nearest station match: 0
Max distance to nearest station (km): 15.2
Mean distance to nearest station (km): 2.66

Schools farthest from any station:
                                            School Name         District  \
2422                             SUNRISE CONVENT SCHOOL           West B   
360                           R.K. International School  North West B-II   
836   Govt. Boys Sr. Sec. School, Tikri Kalan, No. 1...           West B   
837       Government Girls Sr. Sec. School, Tikri Kalan           West B   
834      Sarvodaya Kanya Vidyalaya, Tikri Kalan-1617012           West B   

     nearest_station_name  distance_km  
2422            Aya Nagar       15.200  
360                Bawana       11.335  
836           NSIT Dwarka       10.928  
837           NSIT Dwarka       10.928  
834           NSIT Dwarka       10.925  


In [16]:
schools_with_exposure.to_csv(OUT_DIR / "schools_with_exposure.csv", index=False)
print("Saved:", OUT_DIR / "schools_with_exposure.csv")
schools_with_exposure.head()

Saved: ../data/processed/schools_with_exposure.csv


,District,School Name,Address,SchoolLevel,Latitude,Longitude,Type,nearest_station_id,nearest_station_name,distance_km,...,Ozone (µg/m³)_missing_pct,Benzene (µg/m³)_mean,Benzene (µg/m³)_median,Benzene (µg/m³)_max,Benzene (µg/m³)_missing_pct,Toluene (µg/m³)_mean,Toluene (µg/m³)_median,Toluene (µg/m³)_max,Toluene (µg/m³)_missing_pct,exposure_confidence
0,North West A,The Vivekanand School,"Shiv Mandir Colony, Narela, Delhi-40",Sr.Secondary,28.870837,77.080305,Unaided,STN_029,Narela,5.740,...,0.821918,2.711808,1.99,13.45,0.0,142.138164,125.77,407.23,0.0,Medium
1,North West A,NATIONAL PUBLIC SCHOOL,"SAFIABAD ROAD NARELA, DELHI - 110040",Secondary,28.863631,77.085591,Unaided,STN_029,Narela,4.809,...,0.821918,2.711808,1.99,13.45,0.0,142.138164,125.77,407.23,0.0,High
2,North West A,Singhu-G(Co-Ed)SSS,"Singhu,Delhi",Sr.Secondary,28.861792,77.136931,Govt,STN_001,Alipur,5.399,...,0.547945,1.352959,0.58,52.05,0.0,4.921589,1.71,62.62,0.0,Medium
3,North West A,"Narela,Sect. A-6, Pkt-2 -SBV","GBSSS, Sect. A-6, Pkt-2, Narela, Delhi-110040",Sarvodaya Sr.Secondary,28.856882,77.100818,Govt,STN_029,Narela,3.787,...,0.821918,2.711808,1.99,13.45,0.0,142.138164,125.77,407.23,0.0,High
4,North West A,DAV CENTENARY PUBLIC SCHOOL,NARELA DELHI - 110040,Sr.Secondary,28.856308,77.084533,Unaided,STN_029,Narela,4.092,...,0.821918,2.711808,1.99,13.45,0.0,142.138164,125.77,407.23,0.0,High


##### schools_with_exposure.csv is now your main analysis dataset. Each row contains school identity, location, nearest station, distance to that station, and the station’s pollution summary, so you can compare exposure across districts, school types, and location patterns. This is the dataset you will use for ranking schools, finding underserved areas, and identifying where monitoring gaps exist.

